# Stage 3 Long-Range LoRA Fine-Tuning

Dual-profile notebook for QEVD-FIT-COACH long-range segment caching, stage-3 LoRA training, benchmark prediction generation, and reusable evaluation.

In [1]:
import os
import subprocess
import sys

PROFILE = "cluster_a40"  # switch back to "mac_m3_max" for local sanity runs
use_qlora = False
RUN_BOOTSTRAP = False

packages = [
    "numpy<2",
    "datasets",
    "evaluate",
    "rouge_score",
    "bert-score",
    "nltk",
]
if PROFILE == "cluster_a40" and use_qlora:
    packages.append("bitsandbytes")

if RUN_BOOTSTRAP:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *packages])
else:
    print("Bootstrap skipped. Install if needed:")
    print("%pip install " + " ".join(packages))

Bootstrap skipped. Install if needed:
%pip install numpy<2 datasets evaluate rouge_score bert-score nltk


In [2]:
# # import sys
# # # !pip install opencv-python
# !pip install torchvision

In [3]:
import json
from dataclasses import asdict
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

repo_root = repo_root =Path.cwd().resolve() #Path("/common/home/users/k/kristle.uy.2022/jupyterlab-venv-py-3117/updated_runnable")
if not (repo_root / "src").exists() and (repo_root / "FitCoach" / "src").exists():
    repo_root = repo_root / "FitCoach"
elif not (repo_root / "src").exists() and repo_root.parent.joinpath("src").exists():
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.stage2 import FrozenEfficientNetStreamEncoder, resolve_runtime
from src.stage3 import (
    Stage3LoraConfig,
    Stage3SegmentDataset,
    Stage3TrainingConfig,
    build_segment_feature_cache,
    build_stage3_lora_streamvlm,
    compute_stage3_action_statistics,
    compute_stage3_loss_metrics,
    evaluate_predictions,
    generate_benchmark_predictions,
    generate_segment_prediction,
    load_long_range_segments,
    prepare_stage3_batch,
    save_segment_manifest,
    split_train_validation_segments,
    stage3_collate,
    trace_generation_step_choices,
    train_stage3,
)


/common/home/users/k/kristle.uy.2022/jupyterlab-venv-pytorch-240/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
profile_overrides = {
    "mac_m3_max": {
        "preferred_device": "mps",
        "num_workers": 0,
        "epochs": 20,
        "max_train_segments": 8,
        "max_val_segments": 4,
        "max_benchmark_segments": 8,
        "eval_every_steps": 0,
        "learning_rate": 5e-5,
        "effective_batch_size": 1,
        "log_every": 1,
    },
    "cluster_a40": {
        "preferred_device": "cuda",
        "num_workers": 2,
        "epochs": 15,
        "max_train_segments": None,
        "max_val_segments": 128,
        "max_benchmark_segments": None,
        "eval_every_steps": 0,
        "learning_rate": 1e-6,
        "effective_batch_size": 32,
        "log_every": 10,
    },
}

stage2_checkpoint_dir = Path("/common/home/users/k/kristle.uy.2022/jupyterlab-venv-py-3117/updated_runnable/outputs/stage2_xattn_llama32_cluster_a40_subset30000_e2_validated_instruct_new/final")
stage2_metadata = json.loads((stage2_checkpoint_dir / "stage2_config.json").read_text())
default_vision_checkpoint = repo_root.parent / "ckpts_efficientnet" / "fitness_ally_hypermodel" / "efficientnet4Lite_1.8.3.checkpoint"
metadata_vision_checkpoint = Path(stage2_metadata.get("extra_config", {}).get("vision_checkpoint_path", ""))
vision_checkpoint_path = Path(os.environ.get("FITCOACH_VISION_CHECKPOINT", "")) if os.environ.get("FITCOACH_VISION_CHECKPOINT") else None
if vision_checkpoint_path is None or not vision_checkpoint_path.exists():
    if default_vision_checkpoint.exists():
        vision_checkpoint_path = default_vision_checkpoint
    else:
        vision_checkpoint_path = metadata_vision_checkpoint

runtime = resolve_runtime(preferred_device=profile_overrides[PROFILE]["preferred_device"])
train_metadata_path = "FitCoach/data/combined/feedbacks_long_range_train.json" #repo_root / "data" / "combined" / "feedbacks_long_range_train.json"
benchmark_metadata_path = "FitCoach/data/combined/feedbacks_long_range_benchmark.json"
train_video_dir = "FitCoach/data/combined/long_range_videos_train" #repo_root / "data" / "combined" / "long_range_videos_train"
benchmark_video_dir = "FitCoach/data/combined/long_range_videos_benchmark"
cache_dir = repo_root / "outputs" / f"stage3_segment_cache_{PROFILE}"
run_name = "sanity_overfit_paper_faithful" if PROFILE == "mac_m3_max" else "full_run"
output_dir = repo_root / "outputs" / f"stage3_lora_long_range_{PROFILE}_instruct_new" / run_name
hf_token = "hf_XXXXXXXXXXXXXXXXXXXXXXXXX" #os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
seed = 469
rebuild_cache = False
min_observation_sec = 2.0 if PROFILE == "mac_m3_max" else 0.0

if PROFILE == "mac_m3_max":
    lora_config = Stage3LoraConfig(
        lora_dropout=0.0,
        target_modules=("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"),
        use_qlora=use_qlora,
    )
else:
    lora_config = Stage3LoraConfig(use_qlora=use_qlora)
training_config = Stage3TrainingConfig(
    learning_rate=profile_overrides[PROFILE]["learning_rate"],
    weight_decay=0.01,
    adam_beta1=0.9,
    adam_beta2=0.95,
    grad_clip_norm=1.0,
    epochs=profile_overrides[PROFILE]["epochs"],
    effective_batch_size=profile_overrides[PROFILE]["effective_batch_size"],
    micro_batch_size=1,
    log_every=profile_overrides[PROFILE]["log_every"],
    eval_every_steps=profile_overrides[PROFILE]["eval_every_steps"],
    feedback_end_weight=4.0,
)

print("profile:", PROFILE)
print("runtime:", runtime)
print("stage2 checkpoint:", stage2_checkpoint_dir)
print("vision checkpoint:", vision_checkpoint_path)
print("output dir:", output_dir)
print(asdict(lora_config))
print(asdict(training_config))
print("min observation sec:", min_observation_sec)
print("min feedback open sec:", training_config.min_feedback_open_sec)


profile: cluster_a40
runtime: Stage2RuntimeConfig(device='cuda', llm_dtype=torch.bfloat16, vision_dtype=torch.float32, use_autocast=True)
stage2 checkpoint: /common/home/users/k/kristle.uy.2022/jupyterlab-venv-py-3117/updated_runnable/outputs/stage2_xattn_llama32_cluster_a40_subset30000_e2_validated_instruct_new/final
vision checkpoint: /common/home/users/k/kristle.uy.2022/ckpts_efficientnet/fitness_ally_hypermodel/efficientnet4Lite_1.8.3.checkpoint
output dir: /common/home/users/k/kristle.uy.2022/FitCoach/outputs/stage3_lora_long_range_cluster_a40_instruct_new/full_run
{'r': 32, 'lora_alpha': 32, 'lora_dropout': 0.05, 'bias': 'none', 'target_modules': ('q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'), 'use_qlora': False}
{'learning_rate': 1e-06, 'weight_decay': 0.01, 'adam_beta1': 0.9, 'adam_beta2': 0.95, 'grad_clip_norm': 1.0, 'epochs': 15, 'effective_batch_size': 32, 'micro_batch_size': 1, 'log_every': 10, 'eval_every_steps': 0, 'feedback_action_weight': 

In [5]:
train_segments_all = load_long_range_segments(
    metadata_path=train_metadata_path,
    video_dir=train_video_dir,
    split="train",
)
benchmark_segments_all = load_long_range_segments(
    metadata_path=benchmark_metadata_path,
    video_dir=benchmark_video_dir,
    split="benchmark",
)
train_segments, val_segments = split_train_validation_segments(
    train_segments_all,
    val_fraction=0.1,
    seed=seed,
    max_train_segments=profile_overrides[PROFILE]["max_train_segments"],
    max_val_segments=profile_overrides[PROFILE]["max_val_segments"],
)

benchmark_limit = profile_overrides[PROFILE]["max_benchmark_segments"]
if benchmark_limit is None:
    benchmark_segments = list(benchmark_segments_all)
else:
    benchmark_rng = np.random.default_rng(seed)
    benchmark_indices = benchmark_rng.permutation(len(benchmark_segments_all))[:benchmark_limit].tolist()
    benchmark_segments = [benchmark_segments_all[idx] for idx in benchmark_indices]

print("train segments total:", len(train_segments_all))
print("benchmark segments total:", len(benchmark_segments_all))
print("train segments used:", len(train_segments))
print("val segments used:", len(val_segments))
print("benchmark segments used:", len(benchmark_segments))
print("train sample ids:", [record.segment_id for record in train_segments[:4]])
print("benchmark sample ids:", [record.segment_id for record in benchmark_segments[:4]])
print("benchmark video ids:", sorted({record.video_id for record in benchmark_segments}))
print(train_segments[0].to_dict())


/common/home/users/k/kristle.uy.2022/FitCoach/src/stage3/dataset.py:189: UserWarning: Skipping malformed long-range record because grouped feedback spans do not align with timestamps/transitions: video=0150.mp4, spans=36, timestamps=37, transitions=37
  warnings.warn(


train segments total: 874
benchmark segments total: 398
train segments used: 786
val segments used: 88
benchmark segments used: 398
train sample ids: ['train:0085:004', 'train:0107:001', 'train:0069:001', 'train:0108:001']
benchmark sample ids: ['benchmark:0006:000', 'benchmark:0006:001', 'benchmark:0006:002', 'benchmark:0006:004']
benchmark video ids: ['0006', '0009', '0010', '0011', '0012', '0013', '0014', '0015', '0016', '0017', '0018', '0019', '0023', '0024', '0025', '0026', '0027', '0028', '0029', '0030', '0031', '0032', '0033', '0039', '0040', '0041', '0043', '0045', '0048', '0049', '0050', '0051', '0052', '0053', '0054', '0055', '0056', '0073', '0074', '0077', '0078', '0079', '0087', '0088', '0145', '0146', '0148', '0149', '0154', '0155', '0158', '0165', '0166', '0167', '0168', '0169', '0170', '0171', '0179', '0180', '0181', '0185', '0192', '0193', '0194', '0195', '0196', '0197', '0204', '0205', '0209', '0212', '0214']
{'segment_id': 'train:0085:004', 'split': 'train', 'video_id

In [6]:
vision_encoder = FrozenEfficientNetStreamEncoder(
    checkpoint_path=vision_checkpoint_path,
    device=runtime.device,
    torch_dtype=runtime.vision_dtype,
)

cached_train_segments = build_segment_feature_cache(
    train_segments,
    vision_encoder=vision_encoder,
    cache_dir=cache_dir,
    overwrite=rebuild_cache,
    manifest_path=output_dir / "train_manifest.json",
    progress_label="cache train segments",
)
cached_val_segments = build_segment_feature_cache(
    val_segments,
    vision_encoder=vision_encoder,
    cache_dir=cache_dir,
    overwrite=rebuild_cache,
    manifest_path=output_dir / "val_manifest.json",
    progress_label="cache val segments",
)
cached_benchmark_segments = build_segment_feature_cache(
    benchmark_segments,
    vision_encoder=vision_encoder,
    cache_dir=cache_dir,
    overwrite=rebuild_cache,
    manifest_path=output_dir / "benchmark_manifest.json",
    progress_label="cache benchmark segments",
)

train_dataset = Stage3SegmentDataset(cached_train_segments)
val_dataset = Stage3SegmentDataset(cached_val_segments)
benchmark_dataset = Stage3SegmentDataset(cached_benchmark_segments)
train_dataloader = DataLoader(
    train_dataset,
    batch_size=training_config.micro_batch_size,
    shuffle=True,
    num_workers=profile_overrides[PROFILE]["num_workers"],
    collate_fn=stage3_collate,
)
val_dataloader = DataLoader(
    val_dataset,
    batch_size=training_config.micro_batch_size,
    shuffle=False,
    num_workers=profile_overrides[PROFILE]["num_workers"],
    collate_fn=stage3_collate,
)

seen_train_samples = cached_train_segments[: min(2, len(cached_train_segments))]
benchmark_probe_samples = cached_benchmark_segments[: min(2, len(cached_benchmark_segments))]
eval_probe_sample = cached_val_segments[0] if cached_val_segments else (cached_train_segments[0] if cached_train_segments else None)
optimizer_steps_per_epoch = (len(train_dataloader) + training_config.gradient_accumulation_steps - 1) // training_config.gradient_accumulation_steps
total_optimizer_steps = optimizer_steps_per_epoch * training_config.epochs

print("cache dir:", cache_dir)
print("train batches:", len(train_dataloader))
print("val batches:", len(val_dataloader))
print("gradient_accumulation_steps:", training_config.gradient_accumulation_steps)
print("optimizer steps per epoch:", optimizer_steps_per_epoch)
print("total optimizer steps:", total_optimizer_steps)
print("seen train probes:", [record.segment_id for record in seen_train_samples])
print("benchmark probes:", [record.segment_id for record in benchmark_probe_samples])
print("eval probe sample:", eval_probe_sample.segment_id if eval_probe_sample is not None else None)


cache benchmark segments: 100%|██████████| 398/398 [00:02<00:00, 154.19it/s]

cache dir: /common/home/users/k/kristle.uy.2022/FitCoach/outputs/stage3_segment_cache_cluster_a40
train batches: 786
val batches: 88
gradient_accumulation_steps: 32
optimizer steps per epoch: 25
total optimizer steps: 375
seen train probes: ['train:0085:004', 'train:0107:001']
benchmark probes: ['benchmark:0006:000', 'benchmark:0006:001']
eval probe sample: train:0164:004


In [ ]:
from huggingface_hub import login
login(token="hf_XXXXXXXXXXXXXXXXXXXXXXXXX")

In [8]:
model = build_stage3_lora_streamvlm(
    stage2_checkpoint_dir=stage2_checkpoint_dir,
    device=runtime.device,
    llm_dtype=runtime.llm_dtype,
    hf_token=hf_token,
    lora_config=lora_config,
)

print("model class:", type(model.model))
print("timeline token ids:", model.timeline_token_ids)
print("timeline token strings:", model.timeline_token_strings)
print("paper token mapping:", {
    "<next>": {"token": model.timeline_token_strings["next"], "id": model.timeline_token_ids["next"]},
    "<feedback>": {"token": model.timeline_token_strings["feedback_begin"], "id": model.timeline_token_ids["feedback_begin"]},
    "</feedback>": {"token": model.timeline_token_strings["feedback_end"], "id": model.timeline_token_ids["feedback_end"]},
})
print("trainable params:", sum(parameter.numel() for parameter in model.trainable_parameters()))


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 254/254 [00:01<00:00, 180.98it/s]


model class: <class 'peft.peft_model.PeftModelForCausalLM'>
timeline token ids: {'next': 128002, 'feedback_begin': 128003, 'feedback_end': 128005}
timeline token strings: {'next': '<|reserved_special_token_0|>', 'feedback_begin': '<|reserved_special_token_1|>', 'feedback_end': '<|reserved_special_token_2|>'}
paper token mapping: {'<next>': {'token': '<|reserved_special_token_0|>', 'id': 128002}, '<feedback>': {'token': '<|reserved_special_token_1|>', 'id': 128003}, '</feedback>': {'token': '<|reserved_special_token_2|>', 'id': 128005}}
trainable params: 48627712


In [9]:
def summarize_prediction(prediction: dict[str, object]) -> dict[str, object]:
    summary = {
        "segment_id": prediction["segment_id"],
        "video_id": prediction["video_id"],
        "feedback_count": len(prediction["pred_feedbacks"]),
        "pred_feedbacks": prediction["pred_feedbacks"],
        "pred_feedback_timestamps": prediction["pred_feedback_timestamps"],
        "generated_tokens": prediction["generated_tokens"],
    }
    if "first_feedback_visible_step" in prediction:
        summary["first_feedback_visible_step"] = prediction["first_feedback_visible_step"]
        summary["first_feedback_visible_sec"] = prediction["first_feedback_visible_sec"]
        summary["feedback_blocked_before_threshold"] = prediction["feedback_blocked_before_threshold"]
        summary["min_observation_sec"] = prediction["min_observation_sec"]
        summary["feedback_followed_by_next"] = prediction.get("feedback_followed_by_next")
        summary["cooldown_blocked_immediate_reopen"] = prediction.get("cooldown_blocked_immediate_reopen")
        summary["post_feedback_min_next_steps"] = prediction.get("post_feedback_min_next_steps")
    return summary


def preview_generations(model, records, label: str, max_records: int = 2, min_observation_sec: float = 0.0, post_feedback_min_next_steps: int = 0, return_debug: bool = False):
    predictions = []
    print(label)
    for record in records[:max_records]:
        prediction = generate_segment_prediction(
            model=model,
            sample=record,
            max_feedback_tokens=64,
            max_total_new_tokens=128,
            min_observation_sec=min_observation_sec,
            post_feedback_min_next_steps=post_feedback_min_next_steps,
            return_debug=return_debug,
        )
        predictions.append(prediction)
        print(summarize_prediction(prediction))
    return predictions


model.stage3_min_feedback_open_sec = float(training_config.min_feedback_open_sec)
first_batch = next(iter(train_dataloader))
prepared = prepare_stage3_batch(first_batch, model=model)
train_action_stats = compute_stage3_action_statistics(train_dataloader, model=model)
feedback_action_weight = min(training_config.auto_feedback_weight_cap, train_action_stats["raw_ratio"] ** 0.5) if training_config.feedback_action_weight is None else training_config.feedback_action_weight
for key in ["input_ids", "attention_mask", "vision_xattn_mask", "labels", "action_target_mask", "text_target_mask", "feedback_open_mask", "pre_feedback_next_mask", "feedback_end_mask", "post_feedback_next_mask"]:
    print(key, prepared[key].shape)
print("vision feats:", prepared["vision_feats"]["feats"].shape)
with torch.no_grad():
    outputs = model(
        input_ids=prepared["input_ids"],
        attention_mask=prepared["attention_mask"],
        vision_feats=prepared["vision_feats"],
        vision_xattn_mask=prepared["vision_xattn_mask"],
        labels=None,
    )
loss_metrics = compute_stage3_loss_metrics(
    logits=outputs.logits,
    labels=prepared["labels"],
    action_target_mask=prepared["action_target_mask"],
    text_target_mask=prepared["text_target_mask"],
    feedback_open_mask=prepared["feedback_open_mask"],
    pre_feedback_next_mask=prepared["pre_feedback_next_mask"],
    feedback_end_mask=prepared["feedback_end_mask"],
    post_feedback_next_mask=prepared["post_feedback_next_mask"],
    model=model,
    feedback_action_weight=feedback_action_weight,
    feedback_open_margin=training_config.feedback_open_margin,
    pre_feedback_next_margin=training_config.pre_feedback_next_margin,
    post_feedback_next_margin=training_config.post_feedback_next_margin,
    feedback_open_margin_weight=training_config.feedback_open_margin_weight,
    pre_feedback_next_margin_weight=training_config.pre_feedback_next_margin_weight,
    post_feedback_next_margin_weight=training_config.post_feedback_next_margin_weight,
    feedback_end_weight=training_config.feedback_end_weight,
)
print("finite constrained loss:", bool(torch.isfinite(loss_metrics["loss"]).item()))
print("finite logits:", bool(torch.isfinite(outputs.logits).all().item()))
print("prepared sample:", prepared["samples"][0].segment_id)
print("train action stats:", train_action_stats)
print("chosen feedback_action_weight:", feedback_action_weight)
print("chosen min_feedback_open_sec:", training_config.min_feedback_open_sec)
label_tokens = prepared["labels"][0][prepared["labels"][0] != -100].tolist()
timeline_ids = model.timeline_token_ids
print("supervised token counts:", {
    "total_supervised_tokens": len(label_tokens),
    "action_targets": int(prepared["action_target_mask"][0].sum().item()),
    "text_targets": int(prepared["text_target_mask"][0].sum().item()),
    "feedback_open_targets": int(prepared["feedback_open_mask"][0].sum().item()),
    "pre_feedback_next_targets": int(prepared["pre_feedback_next_mask"][0].sum().item()),
    "feedback_end_targets": int(prepared["feedback_end_mask"][0].sum().item()),
    "post_feedback_next_targets": int(prepared["post_feedback_next_mask"][0].sum().item()),
    "feedback_begin_count": sum(1 for token in label_tokens if token == timeline_ids["feedback_begin"]),
    "feedback_end_count": sum(1 for token in label_tokens if token == timeline_ids["feedback_end"]),
    "next_count": sum(1 for token in label_tokens if token == timeline_ids["next"]),
})
print("loss metrics:", {
    "loss": float(loss_metrics["loss"].item()),
    "ce_loss": float(loss_metrics["ce_loss"].item()),
    "action_loss": float(loss_metrics["action_loss"].item()),
    "text_loss": float(loss_metrics["text_loss"].item()),
    "feedback_open_margin_loss": float(loss_metrics["feedback_open_margin_loss"].item()),
    "pre_feedback_next_margin_loss": float(loss_metrics["pre_feedback_next_margin_loss"].item()),
    "post_feedback_next_margin_loss": float(loss_metrics["post_feedback_next_margin_loss"].item()),
    "feedback_end_loss": float(loss_metrics["feedback_end_loss"].item()),
    "feedback_prob_on_feedback_targets": float(loss_metrics["feedback_prob_on_feedback_targets"].item()),
    "feedback_prob_on_next_targets": float(loss_metrics["feedback_prob_on_next_targets"].item()),
    "feedback_prob_on_feedback_open_targets": float(loss_metrics["feedback_prob_on_feedback_open_targets"].item()),
    "feedback_prob_on_pre_feedback_next_targets": float(loss_metrics["feedback_prob_on_pre_feedback_next_targets"].item()),
    "feedback_prob_on_post_feedback_next_targets": float(loss_metrics["feedback_prob_on_post_feedback_next_targets"].item()),
    "action_count": float(loss_metrics["action_count"].item()),
    "text_count": float(loss_metrics["text_count"].item()),
})
print("decoded supervised target:")
print(model.tokenizer.decode(label_tokens, skip_special_tokens=False))
pretrain_seen_predictions = preview_generations(model, seen_train_samples, label="pre-train seen-train generations")


input_ids torch.Size([1, 232])
attention_mask torch.Size([1, 232])
vision_xattn_mask torch.Size([1, 232])
labels torch.Size([1, 232])
action_target_mask torch.Size([1, 232])
text_target_mask torch.Size([1, 232])
feedback_open_mask torch.Size([1, 232])
pre_feedback_next_mask torch.Size([1, 232])
feedback_end_mask torch.Size([1, 232])
post_feedback_next_mask torch.Size([1, 232])
vision feats: torch.Size([1, 132, 35, 1280])
finite constrained loss: True
finite logits: True
prepared sample: train:0153:001
train action stats: {'next_action_targets': 103449.0, 'feedback_action_targets': 3969.0, 'total_action_targets': 107418.0, 'total_text_targets': 36974.0, 'feedback_open_targets': 3969.0, 'pre_feedback_next_targets': 3969.0, 'feedback_end_targets': 3969.0, 'post_feedback_next_targets': 3969.0, 'raw_ratio': 26.064247921390777}
chosen feedback_action_weight: 5.105315653452857
chosen min_feedback_open_sec: 2.0
supervised token counts: {'total_supervised_tokens': 200, 'action_targets': 139, 't

In [10]:
history = train_stage3(
    model=model,
    dataloader=train_dataloader,
    config=training_config,
    output_dir=output_dir,
    validation_dataloader=val_dataloader,
    probe_sample=eval_probe_sample,
    probe_generation_kwargs={
        "max_feedback_tokens": 64,
        "max_total_new_tokens": 128,
        "post_feedback_min_next_steps": 0,
    },
)
for epoch_record in history:
    print(epoch_record)
    probe = epoch_record.get("epoch_probe")
    if probe is not None:
        print("epoch probe:")
        print({
            "segment_id": probe["segment_id"],
            "video_id": probe["video_id"],
            "feedback_count": probe["feedback_count"],
            "generated_tokens": probe["generated_tokens"],
            "first_feedback_visible_step": probe.get("first_feedback_visible_step"),
            "first_feedback_visible_sec": probe.get("first_feedback_visible_sec"),
            "feedback_opened_before_patience_target": probe.get("feedback_opened_before_patience_target"),
            "feedback_blocked_before_threshold": probe.get("feedback_blocked_before_threshold"),
            "feedback_followed_by_next": probe.get("feedback_followed_by_next"),
            "cooldown_blocked_immediate_reopen": probe.get("cooldown_blocked_immediate_reopen"),
            "action_token_counts": probe["action_token_counts"],
            "feedback_stop_events": probe.get("feedback_stop_events"),
            "pred_feedbacks": probe["pred_feedbacks"],
            "pred_feedback_timestamps": probe["pred_feedback_timestamps"],
        })
        print("epoch probe raw stream text:")
        print(probe["raw_stream_text"])
history


epoch 15/15: 100%|██████████| 786/786 [01:35<00:00,  8.20it/s, action_loss=0.6915, ce_loss=1.8589, feedback_end_loss=8.1450, loss=3.8588, text_loss=4.9326]


{'epoch': 1.0, 'avg_loss': 4.161771589563093, 'avg_ce_loss': 2.1617927120538765, 'avg_action_loss': 0.6914217638908755, 'avg_text_loss': 6.0449800685465185, 'avg_next_action_loss': 0.6913748901460613, 'avg_feedback_action_loss': 0.691661445239118, 'avg_feedback_open_margin_loss': 0.7502087309160306, 'avg_pre_feedback_next_margin_loss': 0.2498248151240458, 'avg_post_feedback_next_margin_loss': 0.24973660146310434, 'avg_feedback_end_loss': 8.730385046878844, 'avg_feedback_action_prob_on_feedback_targets': 0.4999380630770409, 'avg_feedback_action_prob_on_next_targets': 0.4999499331221326, 'avg_feedback_prob_on_feedback_open_targets': 0.4999380630770409, 'avg_feedback_prob_on_pre_feedback_next_targets': 0.49995589764366927, 'avg_feedback_prob_on_post_feedback_next_targets': 0.4999329783306777, 'optimizer_steps': 25.0, 'feedback_action_weight': 5.105315653452857, 'min_feedback_open_sec': 2.0, 'feedback_open_margin': 0.75, 'pre_feedback_next_margin': 0.25, 'post_feedback_next_margin': 0.25, 

[{'epoch': 1.0,
  'avg_loss': 4.161771589563093,
  'avg_ce_loss': 2.1617927120538765,
  'avg_action_loss': 0.6914217638908755,
  'avg_text_loss': 6.0449800685465185,
  'avg_next_action_loss': 0.6913748901460613,
  'avg_feedback_action_loss': 0.691661445239118,
  'avg_feedback_open_margin_loss': 0.7502087309160306,
  'avg_pre_feedback_next_margin_loss': 0.2498248151240458,
  'avg_post_feedback_next_margin_loss': 0.24973660146310434,
  'avg_feedback_end_loss': 8.730385046878844,
  'avg_feedback_action_prob_on_feedback_targets': 0.4999380630770409,
  'avg_feedback_action_prob_on_next_targets': 0.4999499331221326,
  'avg_feedback_prob_on_feedback_open_targets': 0.4999380630770409,
  'avg_feedback_prob_on_pre_feedback_next_targets': 0.49995589764366927,
  'avg_feedback_prob_on_post_feedback_next_targets': 0.4999329783306777,
  'optimizer_steps': 25.0,
  'feedback_action_weight': 5.105315653452857,
  'min_feedback_open_sec': 2.0,
  'feedback_open_margin': 0.75,
  'pre_feedback_next_margin': 

In [11]:
def inspect_action_target_probabilities(model, sample):
    single_batch = stage3_collate([sample.to_dict()])
    prepared_single = prepare_stage3_batch(single_batch, model=model)
    labels = prepared_single["labels"][0]
    input_ids = prepared_single["input_ids"][0]
    attention_mask = prepared_single["attention_mask"][0]
    vision_mask = prepared_single["vision_xattn_mask"][0]
    feedback_begin_id = model.timeline_token_ids["feedback_begin"]
    next_id = model.timeline_token_ids["next"]

    feedback_positions = torch.where(labels == feedback_begin_id)[0]
    next_positions = torch.where(labels == next_id)[0]
    if feedback_positions.numel() == 0 or next_positions.numel() == 0:
        print("Missing supervised action targets for sample", sample.segment_id)
        return None

    feedback_pos = int(feedback_positions[0].item())
    next_candidates = [int(pos.item()) for pos in next_positions if int(pos.item()) < feedback_pos]
    if not next_candidates:
        next_candidates = [int(next_positions[0].item())]
    next_pos = next_candidates[-1]

    def _inspect(prefix_pos: int, label_name: str):
        prefix_input_ids = input_ids[:prefix_pos].unsqueeze(0)
        prefix_attention_mask = attention_mask[:prefix_pos].unsqueeze(0)
        prefix_vision_mask = vision_mask[:prefix_pos].unsqueeze(0)
        visible_steps = int((prefix_vision_mask == 2).sum().item())
        visible_vision_feats = prepared_single["vision_feats"]["feats"][:, :visible_steps]
        with torch.no_grad():
            outputs = model(
                input_ids=prefix_input_ids,
                attention_mask=prefix_attention_mask,
                vision_feats={
                    "feats": visible_vision_feats,
                    "spatial_res": prepared_single["vision_feats"]["spatial_res"],
                },
                vision_xattn_mask=prefix_vision_mask,
                labels=None,
            )
        logits = outputs.logits[0, -1]
        action_logits = torch.stack([logits[next_id], logits[feedback_begin_id]])
        action_probs = torch.softmax(action_logits, dim=0)
        return {
            "label": label_name,
            "target_position": prefix_pos,
            "visible_steps": visible_steps,
            "next_prob": float(action_probs[0].item()),
            "feedback_prob": float(action_probs[1].item()),
            "preferred_action": "<next>" if action_probs[0] >= action_probs[1] else "<feedback>",
        }

    result = {
        "segment_id": sample.segment_id,
        "feedback_target": _inspect(feedback_pos, "feedback_target"),
        "nearby_next_target": _inspect(next_pos, "nearby_next_target"),
    }
    print("action-target probability inspection:")
    print(result)
    return result


def print_raw_prediction_debug(model, sample, label: str, min_observation_sec: float = 0.0, post_feedback_min_next_steps: int = 0):
    debug_prediction = generate_segment_prediction(
        model=model,
        sample=sample,
        max_feedback_tokens=64,
        max_total_new_tokens=128,
        min_observation_sec=min_observation_sec,
        post_feedback_min_next_steps=post_feedback_min_next_steps,
        return_debug=True,
    )
    print(label)
    print("summary:", summarize_prediction(debug_prediction))
    print("first feedback open:", {
        "first_feedback_visible_step": debug_prediction["first_feedback_visible_step"],
        "first_feedback_visible_sec": debug_prediction["first_feedback_visible_sec"],
        "feedback_opened_before_patience_target": (
            debug_prediction["first_feedback_visible_sec"] is not None
            and debug_prediction["first_feedback_visible_sec"] < training_config.min_feedback_open_sec
        ),
        "feedback_blocked_before_threshold": debug_prediction["feedback_blocked_before_threshold"],
        "min_observation_sec": debug_prediction["min_observation_sec"],
        "feedback_followed_by_next": debug_prediction.get("feedback_followed_by_next"),
        "cooldown_blocked_immediate_reopen": debug_prediction.get("cooldown_blocked_immediate_reopen"),
        "post_feedback_min_next_steps": debug_prediction.get("post_feedback_min_next_steps"),
    })
    print("feedback stop events:", debug_prediction.get("feedback_stop_events", []))
    print("action token counts:", debug_prediction["action_token_counts"])
    print("action trace preview:", debug_prediction["action_trace"][:40])
    print("raw stream text:")
    print(debug_prediction["raw_stream_text"])
    return debug_prediction


print("final epoch diagnostics:")
print(history[-1])
posttrain_seen_predictions = preview_generations(
    model,
    seen_train_samples,
    label="post-train seen-train generations (ungated)",
    return_debug=True,
)
posttrain_seen_predictions_cooldown = preview_generations(
    model,
    seen_train_samples,
    label="post-train seen-train generations (ungated + cooldown)",
    post_feedback_min_next_steps=1,
    return_debug=True,
)
posttrain_seen_predictions_gated = preview_generations(
    model,
    seen_train_samples,
    label=f"post-train seen-train generations (gated at {min_observation_sec:.1f}s)",
    min_observation_sec=min_observation_sec,
    return_debug=True,
)
posttrain_benchmark_probe_predictions = preview_generations(
    model,
    benchmark_probe_samples,
    label="post-train benchmark pilot generations (ungated)",
    return_debug=True,
)
posttrain_benchmark_probe_predictions_cooldown = preview_generations(
    model,
    benchmark_probe_samples,
    label="post-train benchmark pilot generations (ungated + cooldown)",
    post_feedback_min_next_steps=1,
    return_debug=True,
)
posttrain_benchmark_probe_predictions_gated = preview_generations(
    model,
    benchmark_probe_samples,
    label=f"post-train benchmark pilot generations (gated at {min_observation_sec:.1f}s)",
    min_observation_sec=min_observation_sec,
    return_debug=True,
)
posttrain_action_inspection = inspect_action_target_probabilities(model, seen_train_samples[0])
raw_seen_debug = print_raw_prediction_debug(
    model,
    seen_train_samples[0],
    label="raw seen-train generation debug",
)
raw_seen_debug_cooldown = print_raw_prediction_debug(
    model,
    seen_train_samples[0],
    label="raw seen-train generation debug (cooldown 1 next)",
    post_feedback_min_next_steps=1,
)
raw_seen_debug_gated = print_raw_prediction_debug(
    model,
    seen_train_samples[0],
    label=f"raw seen-train generation debug (gated at {min_observation_sec:.1f}s)",
    min_observation_sec=min_observation_sec,
)
raw_benchmark_debug = print_raw_prediction_debug(
    model,
    benchmark_probe_samples[0],
    label="raw benchmark generation debug",
)
raw_benchmark_debug_cooldown = print_raw_prediction_debug(
    model,
    benchmark_probe_samples[0],
    label="raw benchmark generation debug (cooldown 1 next)",
    post_feedback_min_next_steps=1,
)
raw_benchmark_debug_gated = print_raw_prediction_debug(
    model,
    benchmark_probe_samples[0],
    label=f"raw benchmark generation debug (gated at {min_observation_sec:.1f}s)",
    min_observation_sec=min_observation_sec,
)

step_trace = trace_generation_step_choices(
    model,
    seen_train_samples[0],
    max_steps=20,
)
print("single-sample raw step trace (ungated):")
print("prompt text:")
print(step_trace["prompt_text"])
for row in step_trace["trace"]:
    print(row)
step_trace_gated = trace_generation_step_choices(
    model,
    seen_train_samples[0],
    max_steps=20,
    post_feedback_min_next_steps=1,
    min_observation_sec=min_observation_sec,
)
print(f"single-sample raw step trace (gated at {min_observation_sec:.1f}s + cooldown 1 next):")
print("prompt text:")
print(step_trace_gated["prompt_text"])
for row in step_trace_gated["trace"]:
    print(row)


final epoch diagnostics:
{'epoch': 15.0, 'avg_loss': 3.8587658635835913, 'avg_ce_loss': 1.8589410464878908, 'avg_action_loss': 0.6915209234215831, 'avg_text_loss': 4.932558901740697, 'avg_next_action_loss': 0.6915409510372249, 'avg_feedback_action_loss': 0.6914182962048752, 'avg_feedback_open_margin_loss': 0.7499403625954199, 'avg_pre_feedback_next_margin_loss': 0.2501143050254453, 'avg_post_feedback_next_margin_loss': 0.2498297849077608, 'avg_feedback_end_loss': 8.14495024668958, 'avg_feedback_action_prob_on_feedback_targets': 0.5000281049929199, 'avg_feedback_action_prob_on_next_targets': 0.5000312354500968, 'avg_feedback_prob_on_feedback_open_targets': 0.5000281049929199, 'avg_feedback_prob_on_pre_feedback_next_targets': 0.5000372170719481, 'avg_feedback_prob_on_post_feedback_next_targets': 0.49995694671575047, 'optimizer_steps': 375.0, 'feedback_action_weight': 5.105315653452857, 'min_feedback_open_sec': 2.0, 'feedback_open_margin': 0.75, 'pre_feedback_next_margin': 0.25, 'post_fee

In [12]:
final_adapter_dir = output_dir / "final_adapter"
model.save_lora_adapter(
    final_adapter_dir,
    extra_config={
        "profile": PROFILE,
        "training_config": asdict(training_config),
        "lora_config": asdict(lora_config),
    },
)
pilot_predictions_path = output_dir / "benchmark_predictions_pilot32.json"
benchmark_pilot_generation_kwargs = {
    "max_records": None,
    "max_feedback_tokens": 64,
    "max_total_new_tokens": 512,
    "max_feedbacks_per_segment": None,
    "min_observation_sec": 0.0,
    "min_feedback_tokens_before_end": 4,
    "soft_feedback_token_budget": 24,
    "feedback_end_logit_bias": 1.5,
    "no_repeat_ngram_size": 3,
    "feedback_repetition_penalty": 1.1,
    "post_feedback_min_next_steps": 0,
}
pilot_predictions = generate_benchmark_predictions(
    model=model,
    records=cached_benchmark_segments,
    output_path=pilot_predictions_path,
    **benchmark_pilot_generation_kwargs,
)
print("saved adapter:", final_adapter_dir)
print("saved pilot predictions:", pilot_predictions_path)
pilot_predictions[:1]


generating benchmark predictions: 100%|██████████| 398/398 [2:38:49<00:00, 23.94s/it, segment_id=benchmark:0214:005]  

saved adapter: /common/home/users/k/kristle.uy.2022/FitCoach/outputs/stage3_lora_long_range_cluster_a40_instruct_new/full_run/final_adapter
saved pilot predictions: /common/home/users/k/kristle.uy.2022/FitCoach/outputs/stage3_lora_long_range_cluster_a40_instruct_new/full_run/benchmark_predictions_pilot32.json


[{'segment_id': 'benchmark:0006:000',
  'video_id': '0006',
  'exercise_name': 'high knees',
  'pred_feedbacks': ["Let's get moving!{lngI see you're starting with a good warm-up. Keep that intensity up! електрон>\nYou're doing great, keep pushing through the burn. Your form is looking solid, but I want to see more engagement in your core. Remember to draw your belly button towards",
   "Let's keep going! You're doing fantastic. Keep that pace and don't forget to breathe. Your form is looking great, but I want to see more power in your movements. Push through the discomfort and give it everything you've got! електрон>\nYou're really pushing yourself now! Keep up",
   "You're doing great! Keep going!{lngI see you're getting close to the end. Give it one more push! електрон>\nYou're almost done! Keep pushing through the pain. You got this!assistant\n\nGreat job! You made it through the tough part! Now,",
   "Let's finish strong! You're doing great! Keep going!{lngI see you're almost done.

In [13]:
metrics_path = output_dir / "benchmark_metrics_pilot32.json"
metrics = evaluate_predictions(
    predictions=pilot_predictions_path,
    references=cached_benchmark_segments,
    output_path=metrics_path,
)
print("saved metrics:", metrics_path)
metrics


[nltk_data] Downloading package wordnet to
[nltk_data]     /common/home/users/k/kristle.uy.2022/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /common/home/users/k/kristle.uy.2022/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /common/home/users/k/kristle.uy.2022/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 840.40it/s]
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.

saved metrics: /common/home/users/k/kristle.uy.2022/FitCoach/outputs/stage3_lora_long_range_cluster_a40_instruct_new/full_run/benchmark_metrics_pilot32.json


{'num_prediction_segments': 398,
 'num_reference_segments': 398,
 'missing_reference_segment_ids': [],
 'meteor': 0.16880793420159285,
 'rougeL': 0.06545189694320108,
 'bert_score': 0.8438029736948655,
 'temporal_f_score': 0.20180995475063598,
 'mean_ttft_sec': 0.045210445374212996,
 'mean_time_to_last_token_sec': 3.0423352421561236,
 'tokens_per_second': 21.409883108585493,
 'token_usage': {'total_prompt_tokens': 13134,
  'total_generated_tokens': 203776,
  'total_tokens': 216910,
  'mean_prompt_tokens': 33.0,
  'mean_generated_tokens': 512.0,
  'mean_total_tokens': 545.0},
 'matched_feedback_examples': [{'segment_id': 'benchmark:0006:000',
   'temporal_fscore_running': 0.4999999999994167,
   'gt_feedback': 'Nice!',
   'pred_feedback': "Let's get moving!{lngI see you're starting with a good warm-up. Keep that intensity up! електрон>\nYou're doing great, keep pushing through the burn. Your form is looking solid, but I want to see more engagement in your core. Remember to draw your bell

In [14]:
%pip install --force-reinstall "numpy<2" evaluate rouge_score bert-score nltk datasets

  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached evaluate-0.4.6-py3-none-any.whl.metadata (9.5 kB)
  Using cached rouge_score-0.1.2-py3-none-any.whl
  Using cached bert_score-0.3.13-py3-none-any.whl.metadata (15 kB)
  Using cached nltk-3.9.4-py3-none-any.whl.metadata (3.2 kB)
  Using cached datasets-4.8.4-py3-none-any.whl.metadata (19 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached pandas-3.0.2-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached requests-2.33.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached xxhash-3.6.0-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (13 kB)
  Using cached multiprocess-0.70.19-py311-none-any.whl.metadata (7.5 kB)
  Using cached fsspec-2026.3.0-py3-none-any.whl.metadata (10 kB)
  Using cached huggingface_hub-1.

In [15]:
!pip install -q "huggingface_hub==1.8.0" "transformers==5.4.0"